## rknn_toolkit2 适配 python3.10, 所以在运行前请更换 kernel

### 安装环境

In [1]:
%pip install torch==2.4.0 torchvision==0.19.0 -i https://pypi.tuna.tsinghua.edu.cn/simple
%pip install setuptools==80.9.0 -i https://pypi.tuna.tsinghua.edu.cn/simple
%pip install onnx==1.18 onnxruntime==1.18 -i https://pypi.tuna.tsinghua.edu.cn/simple
%pip install rknn_toolkit2 -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 9.1 MB/s  0:01:180:00:0100:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 10.0 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 10.1 MB/s  0:00:00m0:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 5.9 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 10.1 MB/s  0:00:02m0:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 5.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 9.8 MB/s  0:00:016m0:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 9.6 MB/s  0:01:040:00:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 10.0 MB/s  0:00:39:00:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 10.6 MB/s  0:00:11:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5

### 数据集 MNIST 处理

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

test_dataset = datasets.MNIST(
    root="./dataset",
    train=False,
    download=True,
    transform=transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1000,
    shuffle=False
)

print("测试集大小:", len(test_dataset))
print("batch 数量:", len(test_loader))

测试集大小: 10000
batch 数量: 10


### ONNX 准确率

In [2]:
import numpy as np
import onnxruntime as ort

def onnxAccuracy(model):
    session = ort.InferenceSession(
        model,
        providers=["CPUExecutionProvider"]
    )

    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    correct = 0
    total = 0

    for images, labels in test_loader:
        images = images.numpy().astype(np.float32)
        outputs = session.run(
            [output_name],
            {input_name: images}
        )[0]
        predictions = np.argmax(outputs, axis=1)
        labels = labels.numpy()
        correct += np.sum(predictions == labels)
        total += labels.shape[0]

    accuracy = correct / total * 100

    print(f"ONNX Accuracy: {accuracy:.2f}%")

onnxAccuracy("./model/model_pruned_fp32.onnx")

ONNX Accuracy: 97.32%


### 校准数据集

In [3]:
import os
from torchvision import datasets

DATASET_DIR = "./dataset"
CALIB_DIR = "./calibration"
DATASET_TXT = "./calibration.txt"

os.makedirs(CALIB_DIR, exist_ok=True)

dataset = datasets.MNIST(
    root=DATASET_DIR,
    train=True,
    download=False
)

num_calib = 200

with open(DATASET_TXT, "w") as f:
    for i in range(num_calib):
        img, label = dataset[i]

        path = os.path.join(
            CALIB_DIR,
            f"{i:05d}.png"
        )

        img.save(path)

        # dataset.txt 每行一张图片
        f.write(path + "\n")

print("校准数据生成完成")
print("图片数量:", num_calib)

校准数据生成完成
图片数量: 200


### INT8 PTQ + 计算准确率

In [4]:
from rknn.api import RKNN
from torchvision import datasets
import numpy as np

ONNX_PATH = "./model/model_pruned_fp32.onnx"
RKNN_PATH = "./model/model_pruned_int8.rknn"
CALIBRATION_PATH = "./calibration.txt"

rknn = RKNN(verbose=False)

ret = rknn.config(
    target_platform="rk3588",
    mean_values=[[0.1307*255]],
    std_values=[[0.3081*255]],
)
if ret != 0:
    print("config failed:", ret)
    exit(ret)

ret = rknn.load_onnx(
    model=ONNX_PATH,
    inputs=["input"],
    input_size_list=[[1, 1, 28, 28]],
)
if ret != 0:
    print("load_onnx failed:", ret)
    exit(ret)

ret = rknn.build(
    do_quantization=True,
    dataset=CALIBRATION_PATH,
)
if ret != 0:
    print("build failed:", ret)
    exit(ret)

ret = rknn.export_rknn(
    RKNN_PATH,
)
if ret != 0:
    print("export_rknn failed:", ret)
    exit(ret)

print("\n########################")
print("\nINT8 PTQ successfully")
print("\n########################")

##################
# RKNN 准确率
##################

test_dataset = datasets.MNIST(
    root="./dataset",
    train=False,
    download=True,
)

ret = rknn.init_runtime()
if ret != 0:
    print("init_runtime failed:", ret)
    exit(ret)

correct = 0
total = len(test_dataset)

for i in range(total):
    image, label = test_dataset[i]
    image = np.asarray(image, dtype=np.uint8)[None, None, :, :]
    
    outputs = rknn.inference(
        inputs=[image],
        data_format=["nchw"],
    )
    
    if outputs is None:
        print(f"Inference failed at sample {i}")
        
    output = np.asarray(outputs[0]).reshape(-1)
    pred = np.argmax(output)

    if pred == label:
        correct += 1

accuracy = correct / total * 100

print("\n########################")
print(f"\nRKNN Accuracy: {accuracy:.2f}%")
print("\n########################")

rknn.release()

/home/hzc/miniconda3/envs/002/lib/python3.10/site-packages/rknn/api/rknn.py:51: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  self.rknn_base = RKNNBase(cur_path, verbose)
I rknn-toolkit2 version: 2.3.2
W load_onnx: If you don't need to crop the model, don't set 'inputs'/'input_size_list'/'outputs'!
I OpFusing 2 : 100%|████████████████████████████████████████████| 100/100 [00:00<00:00, 1497.39it/s]


I Quantizating : 100%|█████████████████████████████████████████████| 11/11 [00:00<00:00, 109.05it/s]
W build: The default input dtype of 'input' is changed from 'float32' to 'int8' in rknn model for performance!
                       Please take care of this change when deploy rknn model with Runtime API!
W build: The default output dtype of 'output' is changed from 'float32' to 'int8' in rknn model for performance!
                      Please take care of this change when deploy rknn model with Runtime API!
I rknn building ...
I rknn building done.
I Target is None, use simulator!



########################

INT8 PTQ successfully

########################


I SessionPreparing : 100%|████████████████████████████████████████| 13/13 [00:00<00:00, 2660.19it/s]



########################

RKNN Accuracy: 97.29%

########################


### 计算内存大小

In [7]:
import os

def get_model_memory_size(path):

    size_bytes = os.path.getsize(path)
    size_mb = size_bytes / 1024 / 1024

    print(f"{path} 模型大小: {size_mb:.2f} MB")

get_model_memory_size(RKNN_PATH)

./model/model_pruned_int8.rknn 模型大小: 0.14 MB
